In [3]:
import nemo
nemo.__version__

'2.6.2'

In [6]:
import nemo.collections.asr as nemo_asr
import nemo.collections.tts as nemo_tts

In [9]:
asr_models = [model for model in dir(nemo_asr.models) if model.endswith("Model")]
asr_models

# tts_models = [model for model in dir(nemo_tts.models) if model.endswith("Model")]
# tts_models

['ASRModel',
 'EncDecCTCModel',
 'EncDecClassificationModel',
 'EncDecDenoiseMaskedTokenPredModel',
 'EncDecDiarLabelModel',
 'EncDecFrameClassificationModel',
 'EncDecHybridRNNTCTCBPEModel',
 'EncDecHybridRNNTCTCModel',
 'EncDecK2RnntSeqModel',
 'EncDecK2SeqModel',
 'EncDecMaskedTokenPredModel',
 'EncDecMultiTalkerRNNTBPEModel',
 'EncDecMultiTaskModel',
 'EncDecRNNTBPEModel',
 'EncDecRNNTModel',
 'EncDecSpeakerLabelModel',
 'SLUIntentSlotBPEModel',
 'SortformerEncLabelModel',
 'SpeechEncDecSelfSupervisedModel']

In [10]:
citrinet = nemo_asr.models.EncDecCTCModelBPE.from_pretrained('stt_en_citrinet_512')

[NeMo I 2026-02-24 22:20:14 nemo_logging:393] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/stt_en_citrinet_512/versions/1.0.0rc1/files/stt_en_citrinet_512.nemo to C:\Users\witze\.cache\torch\NeMo\NeMo_2.6.2\stt_en_citrinet_512\3262321355385bb7cf5a583146117d77\stt_en_citrinet_512.nemo
[NeMo I 2026-02-24 22:20:23 nemo_logging:393] Instantiating model from pre-trained checkpoint
[NeMo I 2026-02-24 22:20:24 nemo_logging:393] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-02-24 22:20:25 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    batch_size: 32
    trim_silence: true
    max_duration: 16.7
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    
[NeMo W 2026-02-24 22:20:25 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: null
    sample_rate: 16000
    batch_size: 32
    shuffle: false
    
[NeMo W 2026-02-24 22:20:25 nemo_logging:405] Please call the ModelPT.setup_test_data() or ModelPT.setup_multiple_test_data() method and provide a valid configuration file to setup the test data l

[NeMo I 2026-02-24 22:20:25 nemo_logging:393] PADDING: 16
[NeMo I 2026-02-24 22:20:26 nemo_logging:393] Model EncDecCTCModelBPE was successfully restored from C:\Users\witze\.cache\torch\NeMo\NeMo_2.6.2\stt_en_citrinet_512\3262321355385bb7cf5a583146117d77\stt_en_citrinet_512.nemo.


In [11]:
citrinet.summarize()

  | Name              | Type                              | Params | Mode 
--------------------------------------------------------------------------------
0 | preprocessor      | AudioToMelSpectrogramPreprocessor | 0      | train
1 | encoder           | ConvASREncoder                    | 36.3 M | train
2 | decoder           | ConvASRDecoder                    | 657 K  | train
3 | loss              | CTCLoss                           | 0      | train
4 | spec_augmentation | SpectrogramAugmentation           | 0      | train
5 | wer               | WER                               | 0      | train
--------------------------------------------------------------------------------
37.0 M    Trainable params
0         Non-trainable params
37.0 M    Total params
147.977   Total estimated model params size (MB)
943       Modules in train mode
0         Modules in eval mode

In [12]:
from omegaconf import OmegaConf
import copy

cfg = copy.deepcopy(citrinet.cfg)
print(OmegaConf.to_yaml(cfg))


sample_rate: 16000
train_ds:
  manifest_filepath: null
  sample_rate: 16000
  batch_size: 32
  trim_silence: true
  max_duration: 16.7
  shuffle: true
  is_tarred: false
  tarred_audio_filepaths: null
validation_ds:
  manifest_filepath: null
  sample_rate: 16000
  batch_size: 32
  shuffle: false
test_ds:
  manifest_filepath:
  - /home/smajumdar/PycharmProjects/nemo-eval/nemo_eval/librispeech/manifests/dev_other.json
  sample_rate: 16000
  batch_size: 32
  shuffle: false
  num_workers: 12
  pin_memory: true
model_defaults:
  repeat: 5
  dropout: 0.0
  separable: true
  se: true
  se_context_size: -1
tokenizer:
  dir: /home/smajumdar/PycharmProjects/nemo-eval/nemo_beta_eval/asrset/manifests/asrset_1.4/tokenizers/no_appen/tokenizer_spe_unigram_v1024/
  type: bpe
preprocessor:
  _target_: nemo.collections.asr.modules.AudioToMelSpectrogramPreprocessor
  sample_rate: 16000
  normalize: per_feature
  window_size: 0.025
  window_stride: 0.01
  window: hann
  features: 80
  n_fft: 512
  frame_s

In [13]:
# OmegaConf won't allow you to add new config items, so we temporarily disable this safeguard.
OmegaConf.set_struct(cfg, False)

# Let's see the old optim config
print("Old Config: ")
print(OmegaConf.to_yaml(cfg.optim))

sched = {'name': 'CosineAnnealing', 'warmup_steps': 1000, 'min_lr': 1e-6}
sched = OmegaConf.create(sched)  # Convert it into a DictConfig

# Assign it to cfg.optim.sched namespace
cfg.optim.sched = sched

# Let's see the new optim config
print("New Config: ")
print(OmegaConf.to_yaml(cfg.optim))

# Here, we restore the safeguards so no more additions can be made to the config
OmegaConf.set_struct(cfg, True)

Old Config: 
name: novograd
lr: 0.05
betas:
- 0.8
- 0.25
weight_decay: 0.001
sched:
  name: CosineAnnealing
  warmup_steps: 1000
  warmup_ratio: null
  min_lr: 1.0e-09
  last_epoch: -1

New Config: 
name: novograd
lr: 0.05
betas:
- 0.8
- 0.25
weight_decay: 0.001
sched:
  name: CosineAnnealing
  warmup_steps: 1000
  min_lr: 1.0e-06

